<a href="https://colab.research.google.com/github/Dires-colab-pro/Monkeypox-disease-stages-dataset/blob/main/unseen_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
DATASET_PATH = "/content/drive/MyDrive/dataset"         # training dataset
DATASET_PATH = "/content/drive/MyDrive/unseen_dataset"  # completely new data

BATCH_SIZE = 32
NUM_CLASSES = 6
EPOCHS = 25
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = ["Macules", "Papules", "Vesicles",
               "Pustules", "Scabs", "Normal"]

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2,0.2,0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

In [ ]:
full_dataset = datasets.ImageFolder(DATASET_PATH)

total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size   = int(0.1 * total_size)
test_size  = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform   = test_transform
test_dataset.dataset.transform  = test_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = models.resnet50(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

model.fc = nn.Sequential(
    nn.Linear(2048, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, NUM_CLASSES)
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)

In [ ]:
def train_model():
    for epoch in range(EPOCHS):
        model.train()
        correct = 0
        train_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()

        train_acc = correct / train_size
        print(f"Epoch {epoch+1}/{EPOCHS} - Train Acc: {train_acc:.4f}")

In [ ]:
train_model()

In [ ]:
y_true, y_pred = [], []

model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu().numpy()

        y_pred.extend(preds)
        y_true.extend(labels.numpy())

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, average='macro'))
print("Recall   :", recall_score(y_true, y_pred, average='macro'))
print("F1 Score :", f1_score(y_true, y_pred, average='macro'))

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.target_layer = target_layer

        self.target_layer.register_forward_hook(self._save_activations)

    def _save_activations(self, module, input, output):
        self.activations = output
        self.activations.retain_grad()

    def generate(self, input_tensor, class_idx):
        self.model.zero_grad()
        output = self.model(input_tensor)

        output[0, class_idx].backward(retain_graph=True)

        # Use the retained gradients from activations
        if self.activations.grad is None:
            raise RuntimeError("Gradients for activations are None. Ensure retain_grad() was called.")

        weights = self.activations.grad.mean(dim=(2,3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1)
        cam = torch.relu(cam)
        cam = cam.squeeze().cpu().detach().numpy()
        cam = cv2.resize(cam, (224,224))
        cam = cam / (cam.max() + 1e-8)
        return cam

In [ ]:
unseen_dataset = datasets.ImageFolder(DATASET_PATH, transform=test_transform)
unseen_loader = DataLoader(unseen_dataset, batch_size=1, shuffle=True)

In [ ]:
def show_unseen_gradcam(img_tensor, true_label):

    model.eval()
    img_tensor = img_tensor.to(DEVICE)

    output = model(img_tensor)
    probs = torch.softmax(output, dim=1)

    pred_class = probs.argmax(1).item()
    confidence = probs[0, pred_class].item()

    gradcam = GradCAM(model, model.layer4)
    cam = gradcam.generate(img_tensor, pred_class)

    img_np = img_tensor.cpu().squeeze().permute(1,2,0).detach().numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())

    plt.figure(figsize=(12,5))

    # Original Image
    plt.subplot(1,2,1)
    plt.imshow(img_np)
    plt.title(f"Image:")
    plt.axis("off")

    # Grad-CAM Focus Region
    plt.subplot(1,2,2)
    plt.imshow(cam, cmap='jet')
    plt.title(f"True Value: {CLASS_NAMES[true_label]}\n"f"Predicted Value: {CLASS_NAMES[pred_class]}\n"
              f"Probability Score: {confidence:.4f}")
    plt.axis("off")

    plt.show()

In [ ]:
for img, label in unseen_loader:
    img.requires_grad_(True) # Ensure gradients can be computed for the input tensor
    show_unseen_gradcam(img, label.item())
    #break   # remove break to test all unseen samples